# Evaluate saved RAG answers

This notebook evaluates `answers.csv` against `data/cancermyth_screening_dataset.json`. Predictions are joined by `question_id`, so CSV row order does not matter and partially completed runs are supported.

In [ ]:
from __future__ import annotations

from collections import defaultdict
import csv
import json
import math
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root from the notebook directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cancermyth_screening_dataset.json"
ANSWERS_PATH = PROJECT_ROOT / "rag-terra" / "rag_run_all" / "answers_oncology_expert.csv"

print(f"Dataset: {DATASET_PATH}")
print(f"Predictions: {ANSWERS_PATH}")

In [ ]:
def parse_boolean(value: str, *, question_id: str) -> bool:
    normalized = value.strip().lower()
    if normalized == "true":
        return True
    if normalized == "false":
        return False
    raise ValueError(f"Answer for question_id={question_id} must be true or false; got {value!r}.")


with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    dataset = json.load(dataset_file)

if not isinstance(dataset, list) or not dataset:
    raise ValueError("The dataset must be a non-empty JSON array.")

dataset_by_id: dict[str, dict] = {}
for record in dataset:
    question_id = str(record.get("id"))
    if question_id in dataset_by_id:
        raise ValueError(f"Duplicate question ID in dataset: {question_id}")
    if not isinstance(record.get("correct_answer"), bool):
        raise ValueError(f"correct_answer for question_id={question_id} must be Boolean.")
    dataset_by_id[question_id] = record

if not ANSWERS_PATH.is_file():
    raise FileNotFoundError(f"No answers file found at {ANSWERS_PATH}. Run the matching run_all_questions_*.ipynb first.")

predictions: dict[str, bool] = {}
with ANSWERS_PATH.open(newline="", encoding="utf-8") as answers_file:
    reader = csv.DictReader(answers_file)
    if reader.fieldnames != ["question_id", "answer"]:
        raise ValueError("answers.csv must contain exactly: question_id, answer")
    for row in reader:
        question_id = row["question_id"].strip()
        if question_id not in dataset_by_id:
            raise ValueError(f"Unknown question_id in answers.csv: {question_id}")
        if question_id in predictions:
            raise ValueError(f"Duplicate question_id in answers.csv: {question_id}")
        predictions[question_id] = parse_boolean(row["answer"], question_id=question_id)

evaluated = [
    {**dataset_by_id[question_id], "prediction": prediction}
    for question_id, prediction in predictions.items()
]

print(f"Dataset questions: {len(dataset):,}")
print(f"Answered questions: {len(evaluated):,}")
print(f"Coverage: {len(evaluated) / len(dataset):.2%}")

In [ ]:
def safe_divide(numerator: int, denominator: int) -> float:
    return numerator / denominator if denominator else math.nan


def calculate_metrics(records: list[dict]) -> dict[str, int | float]:
    tp = sum(row["prediction"] is True and row["correct_answer"] is True for row in records)
    tn = sum(row["prediction"] is False and row["correct_answer"] is False for row in records)
    fp = sum(row["prediction"] is True and row["correct_answer"] is False for row in records)
    fn = sum(row["prediction"] is False and row["correct_answer"] is True for row in records)
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    return {
        "n": len(records),
        "accuracy": safe_divide(tp + tn, len(records)),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": safe_divide(2 * tp, 2 * tp + fp + fn),
        "balanced_accuracy": (recall + specificity) / 2 if not math.isnan(recall) and not math.isnan(specificity) else math.nan,
        "true_positive": tp,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
    }


def format_rate(value: float) -> str:
    return "N/A" if math.isnan(value) else f"{value:.2%}"


def print_metrics(metrics: dict[str, int | float]) -> None:
    print(f"Evaluated:         {metrics['n']:,}")
    for name in ("accuracy", "precision", "recall", "specificity", "f1", "balanced_accuracy"):
        print(f"{name.replace('_', ' ').title():18}{format_rate(metrics[name])}")
    print("\nConfusion matrix counts")
    print(f"True positive:  {metrics['true_positive']:,}")
    print(f"True negative:  {metrics['true_negative']:,}")
    print(f"False positive: {metrics['false_positive']:,}")
    print(f"False negative: {metrics['false_negative']:,}")

In [ ]:
if not evaluated:
    raise ValueError("answers.csv contains no predictions to evaluate.")

overall_metrics = calculate_metrics(evaluated)
print_metrics(overall_metrics)

In [ ]:
def metrics_by_group(records: list[dict], field: str) -> list[dict]:
    groups: dict[str, list[dict]] = defaultdict(list)
    for record in records:
        groups[str(record.get(field) or "Unknown")].append(record)

    rows = []
    for group, group_records in groups.items():
        metrics = calculate_metrics(group_records)
        rows.append({
            field: group,
            "n": metrics["n"],
            "accuracy": round(metrics["accuracy"], 4),
            "precision": round(metrics["precision"], 4),
            "recall": round(metrics["recall"], 4),
            "f1": round(metrics["f1"], 4),
        })
    return sorted(rows, key=lambda row: (-row["n"], row[field]))


print("Metrics by dataset split:")
metrics_by_group(evaluated, "split")

In [ ]:
print("Metrics by cancer type:")
metrics_by_group(evaluated, "cancer")

In [ ]:
mistakes = [
    {
        "question_id": record["id"],
        "cancer": record.get("cancer"),
        "expected": record["correct_answer"],
        "predicted": record["prediction"],
        "question": record["question"],
    }
    for record in evaluated
    if record["prediction"] != record["correct_answer"]
]

print(f"Misclassified questions: {len(mistakes):,}")
mistakes[:20]